# LLM scoring on Kaggle (step D of `04_features.py`)

Runs step D on a Kaggle GPU, with the same code, prompt and parameters as the local
script. The output, `llm_annotations.jsonl`, is copied back into `data/features/`;
step E (agreement with the gold standard) then runs locally without Ollama.

**Before running**

1. From the repository root, build the input archive:
   ```
   zip kaggle_input.zip 04_features.py annotations/gold_guidelines.md data/oil/episodes.csv
   ```
2. On Kaggle: *Datasets → New dataset*, upload `kaggle_input.zip`, keep it private.
3. In this notebook: *Add input* → that dataset. *Settings*: Accelerator **GPU**, Internet **on**
   (Internet requires a verified phone number on the Kaggle account).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Copy the inputs, keeping the repository layout

In [ ]:
import glob, os, pathlib, shutil

src = pathlib.Path(glob.glob("/kaggle/input/**/04_features.py", recursive=True)[0]).parent
repo = pathlib.Path("/kaggle/working/repo")
shutil.copytree(src, repo, dirs_exist_ok=True)
os.chdir(repo)
print(sorted(str(p.relative_to(repo)) for p in repo.rglob("*") if p.is_file()))

## 2. Install and start Ollama

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is up")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

## 3. Model and parameters, read from `04_features.py`

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("features", "04_features.py")
features = importlib.util.module_from_spec(spec)
spec.loader.exec_module(features)
print("model:", features.LLM_MODEL, "| temperature:", features.TEMPERATURE, "| seed:", features.SEED)

In [ ]:
!ollama pull {features.LLM_MODEL}

In [ ]:
print(features.build_prompt())

## 4. Scoring

Resumable: if the session stops, rerun this cell and the episodes already annotated are skipped.

In [ ]:
!python 04_features.py --llm

## 5. Checks

In [ ]:
import pandas as pd

ann = pd.read_json("data/features/llm_annotations.jsonl", lines=True)
episodes = pd.read_csv("data/oil/episodes.csv")
print(f"annotated {ann['episode_id'].nunique()} of {len(episodes)} episodes")
for c, ok in [("direction", ann["direction"].between(-2, 2)),
              ("intensity", ann["intensity"].between(1, 3)),
              ("specificity", ann["specificity"].isin([0, 1]))]:
    print(f"{c:12s} out of range: {(~ok).sum()} | {ann[c].value_counts().sort_index().to_dict()}")
print(open("data/features/llm_prompt.txt").read().split("\n\n")[0])

## 6. Package the output

The archive holds only the LLM output: `episode_features.parquet` stays out, so that
unzipping does not overwrite the local one. Download `llm_output.zip` from the *Output*
panel, unzip it in the repository root, then run locally:
```
python 04_features.py --novelty --llm --agreement
```
`--novelty` keeps the novelty columns in `episode_features.parquet`, which every run of
04 rewrites with the steps it was given.

In [ ]:
import zipfile

with zipfile.ZipFile("/kaggle/working/llm_output.zip", "w") as z:
    for name in ["llm_annotations.jsonl", "llm_prompt.txt"]:
        z.write(f"data/features/{name}", f"data/features/{name}")
    z.write("/kaggle/working/ollama.log", "data/features/ollama.log")
print(zipfile.ZipFile("/kaggle/working/llm_output.zip").namelist())